#### am (test)

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:< rust_operators

## am'

In [ ]:
:= length \* el. (a : array el) : int =
    a |> length

:= index \* el. (i : int) (a : array el) : el =
    backend_switch {
        Gleam = \ () => index a i
        Lua = \ () => index a i
        Fsharp = \ () => index a i
        Python = \ () => $'!a[!i]' : el
        Rust = \ () => index a i
        TypeScript = \ () => index a i
        Zig = \ () => index a i
        Lean = \ () => index a i
        C = \ () => index a i
        Pascal = \ () => index a i
        Wgsl = \ () => index a i
    }

:= index_base \* el. (i : int) (ar : array_base el) : el =
    ar |> array |> index i

:= create_base \* el. (n : int) : array_base el = !!!!ArrayCreate(`el, n)

:= set_base \* el. (ar : array_base el) (i : int) (v : el) : () = !!!!ArrayIndexSet(ar, i, v)

:= length_base \* el. (ar : array_base el) : int =
    ar |> array |> length

#### base_wrappers_round_trip (test)

In [1]:
///- --test --symbol
///> rust

:= a : array_base i32 = create_base 3i32
set_base a 1i32 7i32
(;[ length_base a; index_base 1i32 a ]) |> _assert_eq (;[ 3i32; 7i32 ])

__assert_eq { expected = [3, 7] }

0

In [ ]:
:= base \* el. ((array a) : array el) : array_base el =
    a

:= base' \* el. ((array a) : array el) : array_base el =
    a

:= generic_equable x1 x2 =
    ? length x1 <>.. length x2
    ?> !-
    |:
        :~ ^ 루프 i =
            ? i < length x1
            ?> index i x1 = index i x2 && 루프 (i + 1)
            |: !+
        루프 0

### equable

In [ ]:
:+ append array t = arraym::append

#### append (test)

In [2]:
///- --test --symbol

()

In [ ]:
:= map_base \* t u. (fn : t -> u) (x : array_base t) : array_base u =
    array x
    |> arraym::map fn
    |> \ (array x : array _) => x

:= collect \* t r. (fn : t -> array r) (items : array t) : array r =
    items
    |> arraym::map fn
    |> arraym::fold (++) (;[])

:= init_base n fn : array_base _ =
    arraym::init n fn
    |> \ (array x : array _) => x

:= init n : array_base _ =
    init_base n id

:= choose f l =
    (l, [])
    ||> arraym::foldBack \ x acc =>
        ?= f x ?|
        | Some y => y :: acc
        | None => acc
    |> arraym::fromList

#### choose (test)

In [3]:
///- --test --symbol
///> fsharp
///> cuda
///> typescript
///> cuda

10
|> init
|> \ x => array x : array _
|> choose \ x => ? x % 2 = 0 ?> Some x |: None
|> _assert_eq (;[ 0; 2; 4; 6; 8 ])

.fsx (fsharp)
__assert_eq { expected = [0, 2, 4, 6, 8] }
.py (cuda)
__assert_eq { expected = [0, 2, 4, 6, 8] }
.ts (typescript)
__assert_eq { expected = [0, 2, 4, 6, 8] }
.py (cuda)
__assert_eq { expected = [0, 2, 4, 6, 8] }

In [ ]:
:= sum a =
    a |> arraym::fold (+) 0

#### sum (test)

In [4]:
///- --test --symbol
///> cuda
///> rust
///> typescript

10
|> init
|> \ x => array x : array _
|> sum
|> _assert_eq 45

.py (cuda)
__assert_eq { expected = 45 }
.rs (rust)
__assert_eq { expected = 45 }
.ts (typescript)
__assert_eq { expected = 45 }

0

In [ ]:
:= init_series start end inc : array_base _ =
    := total = conv ((end - start) / inc) + 1
    arraym::init total (conv >> (*) inc >> (+) start)
    |> \ (array x : array _) => x

#### init_series (test)

In [5]:
///- --test --symbol
///> cuda
///> rust
///> typescript

init_series 0i32 6 2
|> array
|> \ x => x : array _
|> _assert_eq (;[ 0i32; 2; 4; 6 ])

.py (cuda)
__assert_eq { expected = [0, 2, 4, 6] }
.rs (rust)
__assert_eq { expected = [0, 2, 4, 6] }
.ts (typescript)
__assert_eq { expected = [0, 2, 4, 6] }

0

In [ ]:
:= head (ar : array _) =
    ? var_is ar || length ar > 0
    ?> ar |> index 0
    |: error_type "The length of the array should be greater than 0."

:= last (ar : array _) =
    := len = length ar
    ? var_is ar || len > 0
    ?> ar |> index (len - 1)
    |: error_type "The length of the array should be greater than 0."

#### last (test)

In [6]:
///- --test --symbol
///> cuda
///> rust
///> typescript

10
|> init
|> \ x => array x : array _
|> last
|> _assert_eq 9

.py (cuda)
__assert_eq { expected = 9 }
.rs (rust)
__assert_eq { expected = 9 }
.ts (typescript)
__assert_eq { expected = 9 }

0

In [ ]:
:= try_pick \* t u. (fn : t -> option u) (array : array t) : option u =
    (array, None)
    ||> arraym::foldBack \ x acc =>
        ?= acc ?|
        | Some _ => acc
        | None => x |> fn

#### try_pick (test)

In [7]:
///- --test --symbol
///> rust
///> cuda
///> typescript
///> cuda

10
|> init
|> \ x => array x : array _
|> try_pick \ x => ? x = 5i32 ?> Some x |: None
|> _assert_eq (Some 5i32)

.rs (rust)
__assert_eq { expected = Some(5) }
.py (cuda)
__assert_eq { expected = Some(5) }
.ts (typescript)
__assert_eq { expected = Some(5) }
.py (cuda)
__assert_eq { expected = Some(5) }

0

In [ ]:
:= indexed \* t u {number}. (ar : array_base t) : array_base (u * t) =
    ((0, ;[]), (array ar : array _))
    ||> arraym::fold \ (i, acc) x =>
        i + 1, acc ++ ;[i, x]
    |> snd
    |> \ (array x : array _) => x

#### indexed (test)

In [8]:
///- --test --symbol
///> rust
///> typescript
///> cuda

arraym::init 3i32 ((*) 2)
|> \ (array x : array _) => x
|> indexed
|> _assert_eq' (base ;[0i32, 0; 1, 2; 2, 4])

.rs (rust)
__assert_eq' { expected = [(0, 0), (1, 2), (2, 4)] }
.ts (typescript)
__assert_eq' { expected = [(0, 0), (1, 2), (2, 4)] }
.py (cuda)
__assert_eq' { expected = [(0, 0), (1, 2), (2, 4)] }

0

In [ ]:
:= slice \* el. from nearTo s : array el =
    arraym::slice { from nearTo } s

#### slice (test)

In [9]:
///- --test --symbol
///> fsharp
///> cuda
///> rust
///> typescript

:= x : array _ = ;[ 1i32; 2; 3 ]
x |> slice 0 0 |> _assert_eq (;[])
x |> slice 0 1 |> _assert_eq (;[ 1 ])
x |> slice 1 1 |> _assert_eq (;[])
x |> slice 1 2 |> _assert_eq (;[ 2 ])
x |> slice 2 2 |> _assert_eq (;[])
x |> slice 0 2 |> _assert_eq (;[ 1; 2 ])

.fsx (fsharp)
__assert_eq { expected = [] }
__assert_eq { expected = [1] }
__assert_eq { expected = [] }
__assert_eq { expected = [2] }
__assert_eq { expected = [] }
__assert_eq { expected = [1, 2] }
.py (cuda)
__assert_eq { expected = [] }
__assert_eq { expected = [1] }
__assert_eq { expected = [] }
__assert_eq { expected = [2] }
__assert_eq { expected = [] }
__assert_eq { expected = [1, 2] }
.rs (rust)
__assert_eq { expected = [] }
__assert_eq { expected = [1] }
__assert_eq { expected = [] }
__assert_eq { expected = [2] }
__assert_eq { expected = [] }
__assert_eq { expected = [1, 2] }
.ts (typescript)
__assert_eq { expected = [] }
__assert_eq { expected = [1] }
__assert_eq { expected = [] }
__assert_eq { expected = [2] }
__assert_eq { expected = [] }
__assert_eq { expected = [1, 2] }

0

In [ ]:
:| range dim =
    | Start : dim
    | End : (() -> dim) -> dim

:= range start end s =
    := start, end =
        := len () =
            s |> length |> conv
        ?= start, end ?|
        | Start start, End fn =>
            start, fn len
        | End start_fn, End end_fn =>
            start_fn len, end_fn len
    s |> slice (start |> convert::unbox) (end |> convert::unbox)

## rust

In [ ]:
:! vec t =
    `(
        rust::rust_type
            `($'Vec<`t>')
            `($'Vec<`t>')
            "type Vec<'T> = class end"
    )

:= from_vec \* el. (vec : vec el) : array el =
    backend_switch {
        Rust = \ () =>
            := vec = ~~ vec
            $'Rc::new(RefCell::new(!vec))' : array el
    }

:= from_vec_base \* el. (vec : vec el) : array_base el =
    backend_switch {
        Rust = \ () =>
            := vec = ~~ vec
            $'Rc::new(RefCell::new(!vec))' : array_base el
    }

:= to_vec \* t. (ab : array_base t) : vec t =
    backend_switch {
        Rust = \ () => $'(!ab).borrow().clone()' : vec t
    }

:= to_vec' \* (t : * -> * -> *) u v. (x : t u v) : vec u =
    backend_switch {
        Rust = \ () => $'(!x).clone()' : vec u
    }

:= to_vec'' \* (t : * -> *) (u : * -> *) v. (x : t (u v)) : vec v =
    backend_switch {
        Rust = \ () => $'(!x).to_vec()' : vec v
    }

:= to_vec''' \* t. (ab : array_base t) : vec t =
    !\\(ab, $'"to_vec($0)"')

:= vec_push \* el. (el : el) (vec : vec el) : vec el =
    := el = ~~ el
    := vec = ~~ vec
    (!\($'"true; let mut !vec = !vec"') : bool) |> ignore
    (!\($'"true; !vec.push(!el)"') : bool) |> ignore
    !\($'"!vec"')

:= vec_reverse \* el. (vec : vec el) : vec el =
    := vec = ~~ vec
    (!\($'"true; let mut !vec = !vec"') : bool) |> ignore
    (!\($'"true; !vec.reverse()"') : bool) |> ignore
    !\($'"!vec"')

:= vec_retain \* el. (fn : el -> bool) (vec : vec el) : vec el =
    := vec = ~~ vec
    := fn = ~~ fn
    (!\($'"true; let mut !vec = !vec"') : bool) |> ignore
    (!\($'"true; !vec.retain(|x| !fn(x.clone()))"') : bool) |> ignore
    !\($'"!vec"')

:= vec_sort_by_key \* el t. (fn : el -> t) (vec : vec el) : vec el =
    := vec = ~~ vec
    := fn = rust::one_param_fn fn
    := fn = ~~ fn
    (!\($'"true; let mut !vec = !vec"') : bool) |> ignore
    (!\($'"true; !vec.sort_by_key(|x| !fn(x.clone()))"') : bool) |> ignore
    !\($'"!vec"')

:= vec_extend \* el. (el : vec el) (vec : vec el) : vec el =
    := el = ~~ el
    := vec = ~~ vec
    (!\($'"true; let mut !vec = !vec"') : bool) |> ignore
    (!\($'"true; !vec.extend(!el)"') : bool) |> ignore
    !\($'"!vec"')

:= vec_mapi \* dim t u. (fn : dim -> t -> u) (ar : vec t) : vec u =
    := fn i = rust::one_param_fn (fn i)
    := fn = ~~ fn
    := ar = ~~ ar
    !\($'"!ar.iter().enumerate().map(|(i, x)| !fn(i.try_into().unwrap())(x.clone())).collect::<Vec<_>>()"')

:= vec_map \* t u. (fn : t -> u) (ar : vec t) : vec u =
    := fn = rust::one_param_fn fn
    !\\((ar, fn), $'"$0.iter().map(|x| $1(x.clone())).collect::<Vec<_>>()"')

:= vec_map' \* t u. (fn : t -> u) (ar : vec t) : vec u =
    := fn = rust::one_param_fn fn
    !\\((ar, fn), $'"$0.iter().map(|x| $1(x.clone())).collect::<Vec<_>>()"')

:= vec_fold' \* t u. (fn : u -> t -> u) (init : u) (ar : vec t) : u =
    (!\\(ar, $'"true; let _vec_fold_ = $0.into_iter().fold(!init, |acc, x| { //"') : bool) |> ignore
    (!\\(fn !\($'"acc"') !\($'"x"'), $'"true; $0 })"') : bool) |> ignore
    !\($'"_vec_fold_"')

:= vec_for_each \* t. (fn : t -> ()) (ar : vec t) : () =
    (!\\((ar, fn), $'"true; $0.iter().for_each(|x| { $1(x.clone()); }); //"') : bool) |> ignore

:= vec_for_each' \* t. (fn : t -> ()) (ar : vec t) : () =
    (!\\(ar, $'"true; $0.into_iter().for_each(|x| { //"') : bool) |> ignore
    (!\\(fn !\($'"x"'), $'$"true"') : bool) |> ignore
    (!\($'"true; }}); //"') : bool) |> ignore

:= vec_for_each'' \* t. (fn : t -> ()) (ar : vec t) : () =
    (!\\(ar, $'"true; $0.into_iter().for_each(|x| { //"') : bool) |> ignore
    (!\\(fn !\($'"x"'), $'$"true"') : bool) |> ignore
    (!\($'"true; }}); //"') : bool) |> ignore

:= vec_for_each''' \* t. (fn : t -> ()) (ar : vec t) : () =
    (!\\(ar, $'"true; $0.into_iter().for_each(|x| { //"') : bool) |> ignore
    (!\\(fn !\($'"x"'), $'$"true"') : bool) |> ignore
    (!\($'"true; }); //"') : bool) |> ignore

:= vec_filter \* t. (fn : t -> bool) (ar : vec t) : vec t =
    := fn = rust::one_param_fn fn
    !\\((ar, fn), $'"$0.iter().filter(|x| $1((*x).clone())).cloned().collect::<Vec<_>>()"')

:= vec_len \* t. (vec : vec t) : convert::unativeint =
    backend_switch {
        Rust = \ () => $'((!vec).len() as usize)' : convert::unativeint
    }

:= vec_item \* t. (i : convert::unativeint) (vec : vec t) : t =
    backend_switch {
        Rust = \ () => $'(!vec)[!i].clone()' : t
    }

:= vec_ref_len \* t. (vec : rust::ref (rust::mut' (vec t))) : convert::unativeint =
    backend_switch {
        Rust = \ () => $'(!vec).len()' : convert::unativeint
    }

:= vec_extend_from_slice \* t. (items : vec t) (vec : rust::ref (rust::mut' (vec t))) : () =
    backend_switch {
        Rust = \ () => $'(!vec).extend_from_slice(&(!items))' : ()
    }

:= vec_ref_to_array \* t. (vec : rust::ref (rust::mut' (vec t))) : array t =
    backend_switch {
        Rust = \ () => $'Rc::new(RefCell::new((!vec).clone()))' : array t
    }

:= vec_extend_from_array \* t. (items : array t) (vec : rust::ref (rust::mut' (vec t))) : () =
    backend_switch {
        Rust = \ () => $'(!vec).extend_from_slice(&(!items).borrow())' : ()
    }

:= vec_drain_front \* t. (count : convert::unativeint) (vec : rust::ref (rust::mut' (vec t))) : vec t =
    backend_switch {
        Rust = \ () => $'(!vec).drain(..!count).collect::<Vec<_>>()' : vec t
    }

:= vec_chunks \* t. (n : i32) (vec : vec t) : vec (vec t) =
    !\\(vec, $'"$0.chunks(!n).map(|x| x.into_iter().map(|x| x.clone()).collect::<Vec<_>>()).collect::<Vec<_>>()"')

:! slice t =
    `(
        rust::rust_type
            `($'Slice<`t>')
            `($'[`t]')
            "type Slice<'T> = class end"
    )

:== slice'_switch el =
    {
        Gleam : $'Slice\'<`el>'
        Lua : $'Slice\'<`el>'
        Fsharp : $'Slice\'<`el>'
        Python : $'Slice\'<`el>'
        TypeScript : $'Slice\'<`el>'
        Cpp : $'Slice\'<`el>'
        CppHost : $'Slice\'<`el>'
        Pascal : $'Slice\'<`el>'
        Cuda : $'Slice\'<`el>'
        Rust : $'_'
    }
:! slice' el dim =
    `(
        rust::fsharp_global "type Slice'<'T> = class end"
        $'' : $'backend_switch `(slice'_switch el)'
    )

:! slice'' el dim =
    `(
        rust::rust_type
            `($'Slice\'_<`el>')
            `($'[`el; 10]')
            "type Slice'_<'T> = class end"
    )

:= slice_singleton \* dim el. (x : option el) : slice' el dim =
    ?= x ?|
    | Some x => !\($'"[!x]"')
    | None =>
        !\($'"[\\\"\\\".to_string()]"') : slice' el dim

:= slice_count \* t dim. (x : slice' t dim) : convert::unativeint =
    !\($'"!x.len()"')

:= slice_range \* t dim.
    (start : range t) (end : range t) (s : slice' t dim) : rust::ref (slice' t dim)
    =
    := len () =
        s |> slice_count
    := start, (end : convert::unativeint) =
        ?= start, end ?|
        | Start start, End fn => start, (len >> convert::convert) |> fn |> convert::convert
        | End start_fn, End end_fn =>
            (len >> convert::convert) |> start_fn, (len >> convert::convert) |> end_fn |> convert::convert
    ?= start, end ?|
    | start, end ?& end |> convert::unbox =. len () => !\($'"&!s[!start..]"')
    | start, end => !\\((start, end), $'"&!s[$0..$1]"')

:= new_slice \* el dim. (el : el) : slice' el dim =
    !\\(el, $'"[$0; @dim as usize]"')

:= as_slice \* t. (x : array_base t) : rust::ref (slice t) =
    := x = x |> to_vec
    !\($'"!x.as_slice()"')

:= slice_to_vec \* t. (slice : rust::ref (slice t)) : vec t =
    !\\(slice, $'"$0.iter().map(|x| *x).collect::<Vec<_>>()"')

:= to_le_bytes \* t. (x : t) : slice' u8 8 =
    backend_switch {
        Rust = \ () => $'(!x).to_le_bytes().to_vec()' : slice' u8 8
    }

:= as_bytes \* t. (x : t) : rust::ref (slice u8) =
    backend_switch {
        Rust = \ () => $'(!x).as_bytes()' : rust::ref (slice u8)
    }

:= any \* t. (fn : t -> bool) (source : array_base t) : bool =
    := n = source |> array |> length
    :~ ^ loop i =
        ? i >= n ?> !-
        |? source |> index_base i |> fn ?> !+
        |: loop (i + 1)
    loop 0

#### any (test)

In [10]:
///- --test --symbol
///> rust
///> typescript

:= source = init 5
(source |> any (\ x => x = 3), source |> any (\ x => x > 10)) |> _assert_eq (!+, !-)

.rs (rust)
__assert_eq { expected = (true, false) }
.ts (typescript)
__assert_eq { expected = (true, false) }

0

### iter_collect vec

In [ ]:
:+ iter_collect vec = \ (iter : into_iterator u) =>
    !\\(iter, $'"$0.collect::<Vec<_>>()"')

:+ iter_collect'' vec = \ (iter : into_iterator (t (u v))) =>
    !\\(iter, $'"$0.collect::<Vec<_>>()"')

:= new_vec \* t. (items : list t) : vec t =
    backend_switch {
        Rust = \ () => items |> listm::fold (\ v x => vec_push x v) ($'Vec::new()' : vec t)
    }

#### new_vec (test)

In [11]:
///- --test --symbol
///> rust

[ 0i32; 1 ]
|> new_vec
|> sm'::format_debug'
|> sm'::from_std_string
|> _assert_eq "[0, 1]"

__assert_eq { expected = [0, 1] }

0

#### new_vec_keeps_a_list_built_at_runtime (test)

In [12]:
///- --test --symbol
///> rust

:~ ^ build (n : i32) (acc : list i32) : list i32 = ? n <= 0i32 ?> acc |: build (n - 1i32) (n :: acc)
:= ~count = 3i32
build count [] |> new_vec |> vec_len |> convert::i32 |> _assert_eq 3i32

__assert_eq { expected = 3 }

0

## fsharp

In [ ]:
:= average \* el {number}. (a : array el) : el =
    run_target \|
        | Fsharp => \ () => $'!a |> Array.average' : el
        | _ => \ () => arraym::fold (+) 0 a / conv (length a)

#### average_divides_the_sum_by_the_count (test)

In [13]:
///- --test --symbol
///> rust

;[ 1.0f64; 2.0f64; 4.5f64 ] |> average |> _assert_eq 2.5f64

__assert_eq { expected = 2.5 }

0

In [ ]:
:= distinct \* el. (a : array el) : array el =
    ? is_rust () ?>
        := a = $'!a ' : array el
        := r = $'Rc::new(RefCell::new(Vec::with_capacity(!a.borrow().len())))' : array el
        loop::for' { from = 0; nearTo = length a } \ i =>
            := x = index i a
            := n = length r
            := s = mut { j = 0 }
            loop::while' (\ () => s::j < n && index s::j r <> x) \ () =>
                s <- { j = s::j + 1 }
            ? s::j = n ?> $'!r.borrow_mut().push(!x)' : ()
        $'!r ' : array el
    |: $'!a |> Array.distinct'

:= growable \* el. () : array el =
    run_target \|
        | Rust => \ () => $'Rc::new(RefCell::new(Vec::new()))' : array el
        | _ => \ () => failwith "growable: no arm for this backend"

:= push_back \* el. (item : el) (items : array el) : () =
    run_target \|
        | Rust => \ () => $'(!items).borrow_mut().push(!item)' : ()
        | _ => \ () => failwith "push_back: no arm for this backend"

:= skip \* el. (n : int) (a : array el) : array el =
    run_target \|
        | Rust => fun () => $'Rc::new(RefCell::new((!a).borrow().iter().skip((!n) as \
            usize).cloned().collect::<Vec<_>>()))' : array el
        | Zig => \ () => $'(spiral_gpa.dupe(`el, (!a)[@@intCast(!n)..]) catch @@panic("out of memory"))' : array el
        | _ => \ () => $'!a |> Array.skip !n '

:= skip_while \* el. (fn : el -> bool) (a : array el) : array el =
    ? is_rust ()
    ?> $'Rc::new(RefCell::new((!a).borrow().iter().skip_while(|x| (!fn)((*x).clone())).cloned().collect::<Vec<_>>()))'
    |: $'!a |> Array.skipWhile !fn '

:= to_list_base' \* t. (items : array_base t) : listm'::list' t =
    backend_switch {
        Gleam = \ () => $'!items |> spiral_array_to_list' : listm'::list' t
        Lua = \ () =>
            := len = $'##(!items)' : i32
            := r : array t = arraym::init len \ i =>
                := at = i + 1i32
                $'!items[!at]' : t
            := (array r) = r
            $'!r ' : listm'::list' t
        Fsharp = \ () => $'!items |> Array.toList' : listm'::list' t
        Python = \ () => $'list(!items)' : listm'::list' t
        Rust = \ () => $'Rc::new((!items).borrow().clone())' : listm'::list' t
        TypeScript = \ () => $'[...!items]' : listm'::list' t
        Zig = \ () => $'!items ' : listm'::list' t
    }

:= to_list' \* t. (items : array t) : listm'::list' t =
    items |> base |> to_list_base'

#### to_list (test)

In [14]:
///- --test --symbol
///> cuda
///> rust
///> typescript

;[ -3i32; 6 ]
|> to_list'
|> listm'::unbox
|> _assert_eq [ -3; 6 ]

.py (cuda)
__assert_eq { expected = Cons(-3, Cons(6, Nil)) }
.rs (rust)
__assert_eq { expected = Cons(-3, Cons(6, Nil)) }
.ts (typescript)
__assert_eq { expected = Cons(-3, Cons(6, Nil)) }

0

#### to_list_snapshots_the_array (test)

In [15]:
///- --test --symbol
///> lua
///> python
///> rust
///> typescript

:= source = ;[ 1i32; 2i32; 3i32 ]
:= snapshot = to_list' source
set source 0i32 9i32
snapshot |> listm'::unbox |> _assert_eq [ 1i32; 2i32; 3i32 ]

.lua (lua)
__assert_eq { expected = Cons(1, Cons(2, Cons(3, Nil))) }
.py (python)
__assert_eq { expected = Cons(1, Cons(2, Cons(3, Nil))) }
.rs (rust)
__assert_eq { expected = Cons(1, Cons(2, Cons(3, Nil))) }
.ts (typescript)
__assert_eq { expected = Cons(1, Cons(2, Cons(3, Nil))) }

0

In [ ]:
:= vec_collect fn vec =
    ((;[] |> base |> to_vec), ((vec |> from_vec : array _) |> to_list' |> listm'::unbox))
    ||> listm::fold \ acc x =>
        acc |> vec_extend (fn x)

:= vec_collect_option vec =
    ((;[] |> base |> to_vec |> Ok), ((vec |> from_vec : array _) |> arraym::toList))
    ||> listm::fold \ acc x =>
        x
        |> resultm::unbox
        |> \ x =>
            ?= acc, x |> resultm::map optionm'::unbox ?|
            | Ok acc, Ok (Some x) => acc |> vec_extend x |> Ok
            | _, Error error => error |> Error
            | _ => acc

:= vec_collect_into \* (c : * -> * -> *) t e.
    (x : vec (c t e))
    : c (vec t) e
    =
    !\($'"!x.into_iter().collect()"')

:= parallel_map \* el el'. (fn : el -> el') (a : array el) : array el' =
    run_target \|
        | Fsharp => \ () => $'!a |> Array.Parallel.map !fn ' : array el'
        | _ => \ () => arraym::map fn a

:= map' \* el el'. (fn : el -> el') (a : array el) : array el' =
    run_target \|
        | Fsharp => \ () => $'!a |> Array.map !fn ' : array el'
        | _ => \ () => arraym::map fn a

:= sort_by \* el key. (fn : el -> key) (a : array el) : array el =
    := fsharp () = $'!a |> Array.sortBy !fn ' : array el
    := insertion_sort () =
        := r = arraym::map id ($'!a ' : array el)
        := keys = arraym::map fn ($'!a ' : array el)
        loop::for' { from = 1; nearTo = length r } \ i =>
            := x = index i r
            := k = index i keys
            := s = mut { j = i }
            loop::while' (\ () => s::j > 0 && gt_is (comp (index (s::j - 1) keys) k)) \ () =>
                := j = s::j
                set r j (index (j - 1) r)
                set keys j (index (j - 1) keys)
                s <- { j = j - 1 }
            set r s::j x
            set keys s::j k
        $'!r ' : array el
    backend_switch {
        Gleam = \ () => fsharp ()
        Lua = insertion_sort
        Fsharp = \ () => fsharp ()
        Python = \ () =>
            := keys = arraym::map fn ($'!a ' : array el)
            $'!a[sorted(range(len(!a)), key=lambda __i: (lambda __k: (__k.tag, __k) if hasattr(__k, "tag") else \
                __k)(!keys[__i]))]' : array el
        TypeScript = \ () =>
            := keys = arraym::map fn ($'!a ' : array el)
            global (
                "function spiral_compare(a: any, b: any): number { if (typeof a === 'object' && a !== null && "
                ^^ "typeof b === 'object' && b !== null) { if (Array.isArray(a)) { for (let i = 0; i < "
                ^^ "Math.min(a.length, b.length); i++) { const c = spiral_compare(a[i], b[i]); if (c !== 0) return c; "
                ^^ "} return a.length - b.length; } if (a.tag !== b.tag) return a.tag - b.tag; for (const k of "
                ^^ "Object.keys(a)) { if (k === 'tag') continue; const c = spiral_compare(a[k], b[k]); if (c !== 0) "
                ^^ "return c; } return 0; } return a < b ? -1 : a > b ? 1 : 0; }"
            )
            $'!a.map((_x: any, i: number) => i).sort((i: number, j: number) => spiral_compare(!keys[i], \
                !keys[j])).map((i: number) => !a[i])' : array el
        Cpp = \ () => fsharp ()
        CppHost = \ () => fsharp ()
        Pascal = \ () => fsharp ()
        Cuda = \ () => fsharp ()
        Zig = insertion_sort
        Lean = insertion_sort
        Rust = \ () =>
            := a = $'!a ' : array el
            := key_fn = ~~ (rust::one_param_fn fn)
            := r = $'Rc::new(RefCell::new(!a.borrow().clone()))' : array el
            := keys =
                $'Rc::new(RefCell::new(!a.borrow().iter().map(|x| !key_fn(x.clone())).collect::<Vec<_>>()))' : array key
            loop::for' { from = 1; nearTo = length a } \ i =>
                := x = index i r
                := k = index i keys
                := s = mut { j = i }
                loop::while' (\ () => s::j > 0 && gt_is (comp (index (s::j - 1) keys) k)) \ () =>
                    := j = s::j
                    set r j (index (j - 1) r)
                    set keys j (index (j - 1) keys)
                    s <- { j = j - 1 }
                set r s::j x
                set keys s::j k
            $'!r ' : array el
    }

:= sort \* el. (a : array el) : array el =
    run_target \|
        | Fsharp => \ () => $'!a |> Array.sort'
        | _ => \ () => sort_by id a

:= sort_descending \* el. (a : array el) : array el =
    run_target \|
        | Fsharp => \ () => $'!a |> Array.sortDescending'
        | _ => \ () =>
            := ascending = sort a
            := n = length ascending
            arraym::init n \ i => index (n - 1 - i) ascending

#### sort_orders_on_every_backend (test)

In [16]:
///- --test --symbol
///> rust
///> python
///> typescript
///> lua

:= a = ;[ 3i32; 1i32; 2i32 ]
:= ascending = sort a
? index 0 ascending <> 1i32 || index 2 ascending <> 3i32 ?> failwith "ascending"
:= descending = sort_descending a
? index 0 descending <> 3i32 || index 2 descending <> 1i32 ?> failwith "descending"

.rs (rust)

.py (python)

.ts (typescript)

.lua (lua)


0

In [ ]:
:= transpose \* el. (a : array_base (array_base el)) : array_base (array_base el) =
    run_target \|
        | Fsharp => \ () => $'!a |> Array.transpose' : array_base (array_base el)
        | _ => \ () =>
            := rows : array (array_base el) = array a
            := height : int = length rows
            := width : int = ? height = 0 ?> 0 |: length (array (index 0 rows) : array el)
            arraym::init width (\ (c : int) =>
                arraym::init height \ (r : int) => index c (array (index r rows) : array el)
                |> \ (array column : array el) => column
            )
            |> \ (array result : array (array_base el)) => result

#### transpose_swaps_rows_and_columns (test)

In [17]:
///- --test --symbol
///> rust

:= row = \ (cells : array i32) => cells |> \ (array x : array i32) => x
:= rows : array (array_base i32) = ;[ row ;[ 1i32; 2i32; 3i32 ]; row ;[ 4i32; 5i32; 6i32 ] ]
:= flipped : array (array_base i32) = array (rows |> \ (array x : array (array_base i32)) => x |> transpose)
:= last : array i32 = array (index 2 flipped)
(length flipped, index 0 last, index 1 last) |> _assert_eq (3, 3i32, 6i32)

__assert_eq { expected = (3, 3, 6) }

0

In [18]:
:= try_item \* el. (i : i32) (a : array el) : option el =
    := fsharp () : option el = $'!a |> Array.tryItem !i ' |> optionm'::unbox
    := in_bounds_item () : option el =
        := a = $'!a ' : array el
        ? i >= 0 && i < length a ?> Some (index i a) |: None
    backend_switch {
        Gleam = fsharp
        Lua = fsharp
        Fsharp = fsharp
        Python = fsharp
        TypeScript = fsharp
        Cpp = fsharp
        CppHost = fsharp
        Pascal = fsharp
        Cuda = fsharp
        Zig = in_bounds_item
        Lean = in_bounds_item
        Rust = in_bounds_item
    }

#### try_item_is_none_out_of_bounds (test)

In [19]:
///- --test --symbol
///> rust
///> zig

:= a = ;[ 10i32; 20i32 ]
:= code (i : i32) : i32 = ?= try_item i a ?| | Some x => x | None => -1i32
[ code (-1i32); code 0i32; code 1i32; code 2i32 ] |> _assert_eq [ -1i32; 10i32; 20i32; -1i32 ]

.rs (rust)
__assert_eq { expected = Cons(-1, Cons(10, Cons(20, Cons(-1, Nil)))) }
.zig (zig)
__assert_eq { expected = Cons(-1, Cons(10, Cons(20, Cons(-1, Nil)))) }

0